# Op3nSrc — Recommender demo (GitHub user → repos → issues)

What the app does when a user enters their GitHub username (`overall-idea.md`):

1. **Profile** — the user's GitHub data (own repos, languages, merged PRs) → one text
   (`recommender/github_profile.py`). Forks are skipped.
2. **Repos** — the trained classifier (best model in `classification.ipynb`) gives a probability for every
   repo → **top 5**.
3. **Issues** — inside each of those repos, open issues are ranked by TF-IDF similarity to the profile
   (+ small boost for `good first issue` / `help wanted`) → **top 3**.
4. **Why** — the words the profile and the repo/issue share.

Users come from `dummy_users/` (same structure as the GitHub API) until the live fetch exists.

In [1]:
import sys
from pathlib import Path

# make the project root importable whether the notebook runs from the root or notebooks/
ROOT = Path.cwd() if (Path.cwd() / "recommender").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from recommender.data import load_all
from recommender.classification import build_dataset
from recommender.models import make_classifiers
from recommender.github_profile import list_users, load_user, build_profile
from recommender.recommend import Recommender, show

tables = load_all()
df = build_dataset(tables)

# final model: the cross-validation winner in classification.ipynb, retrained on ALL developers
classifier = make_classifiers(include_sbert=False)["TF-IDF + Naive Bayes"]
classifier.fit(df.text, df.repo)

recommender = Recommender(classifier, tables["repositories"], tables["issues"])
print(f"classifier knows {len(classifier.classes_)} repos; "
      f"{recommender.class_mask.sum()} of them have open issues and can be recommended")

classifier knows 117 repos; 116 of them have open issues and can be recommended


## 1. The dummy users

In [2]:
pd.DataFrame([build_profile(load_user(u))[1] for u in list_users()]).set_index("login")

,own repos,forks skipped,top languages,merged PRs
login,,,,
demo-alex-py,6,1,"python, html, shell, lua, css",3
demo-jordan-web,4,0,"css, javascript, html, typescript",0
demo-sam-rust,5,0,"rust, go, html, makefile, shell",3


## 2. Recommendations

In [3]:
show(recommender.recommend_user("demo-alex-py"))

Recommendations for demo-alex-py  (languages: python, html, shell, lua, css; merged PRs: 3)

1. encode/django-rest-framework
   Why: django, rest, api, settings, encode
   Issues to start with:
     • #3460 Document that explicitly declared fields will not have their attribute
     • #9179 Not possible to override serializer_field_mappings with a field that h
     • #8273 `required: True` set on a nullable field which is included in combined

2. litestar-org/litestar
   Why: sqlalchemy, pydantic, pytest, asyncio, api
   Issues to start with:
     • #2118 Enhancement: Revoke jwt token  [good first issue, help wanted]
     • #3362 Docs: Move `.. code-block::` directives into `docs/examples/`  [help wanted]
     • #2015 Enhancement: Support for Typing the `query` keyword (and other reserve  [good first issue, help wanted]

3. python/cpython
   Why: python, test, tests, testing, html
   Issues to start with:
     • #158102 Add C API tests for the slice C API
     • #158053 IDLE on macOS 27

In [4]:
show(recommender.recommend_user("demo-sam-rust"))

Recommendations for demo-sam-rust  (languages: rust, go, html, makefile, shell; merged PRs: 3)

1. tokio-rs/tokio
   Why: tokio, rust, chat, networking, tcp
   Issues to start with:
     • #3974 tokio::select! cfg-gated  [help wanted]
     • #8312 Test failing: `test_oneshot_creates_span`
     • #3018 io::driver should have loom tests  [help wanted]

2. kubernetes/kubernetes
   Why: kubernetes, help, values, code
   Issues to start with:
     • #142208 kubectl describe pod does not show env vars sourced from valueFrom.fil
     • #142374 kube-apiserver keep using a stale webhook configuration after an updat
     • #142072 resource quota controller leaks informer event handlers when a monitor

3. apache/airflow
   Why: kubernetes, provider, experimental, help, built
   Issues to start with:
     • #73006 KubernetesPodOperator task is marked SUCCESS when the task process rec
     • #72592 Support Managed Knowledge Bases in BedrockCreateKnowledgeBaseOperator  [good first issue]
     • #728

In [5]:
show(recommender.recommend_user("demo-jordan-web"))

Recommendations for demo-jordan-web  (languages: css, javascript, html, typescript; merged PRs: 0)

1. expo/expo
   Why: react, app, javascript, typescript, api
   Issues to start with:
     • #50810 console.log() displays JavaScript object keys with quotes, making outp
     • #50362 [expo-localization][android] useCalendars() does not re-render when th
     • #50465 [expo][dom][iOS] OTA DOM base URL can select and cache an embedded ass

2. sveltejs/svelte
   Why: javascript, website
   Issues to start with:
     • #18704 Svelte Inspector is incompatible with previous Vite transform plugin
     • #18802 Hydration mismatch in <svelte:head> can remove unrelated stylesheet vi
     • #18799 Async component initialization crashes when current_batch is null

3. nodejs/node
   Why: javascript, website, api, using, list
   Issues to start with:
     • #65930 docs: suggest adding performance comparison table for fs.promises vs c
     • #65974 SQLite in core: maintenance trade-offs compared with

## Notes

- Repos are shown by **rank** only: Naive Bayes ranks repos well, but its probability values are not
  meaningful (all close to 1/117), so showing them would mislead.
- The classifier is trained on PR text and issue titles, while a new user's profile is mostly their own repos
  and languages — a known difference between training and app input (limitation for the report). It shows most
  for users without merged PRs (`demo-jordan-web`).
- Only repos the classifier knows (≥ 5 developers in our data) **and** that have open issues are recommended.